<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-02-write-your-own-autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 (graded) — Write your own autograd
**Course 1: Hands-On Deep Learning with Python — Chapter 2: Backprop & a tiny autograd engine**

**Problem brief (Leo Farkas, Meridian Bank, continued):** "A forward pass isn't a model.
Make it learn."

**What you'll submit:**
1. Working `backward` for `Linear`, `ReLU`, `Sigmoid`, `BCELoss`, passing the gradient-check
   harness below.
2. A trained MLP (hand-written SGD) that beats the bank's AUC of 0.77 on held-out applicants.
3. A training-curve plot and a short analysis.
4. A scorecard with a slice metric, and the decision-dossier-lite half page.

## 1. Load and split the data

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd

np.random.seed(0)

def load_credit_data():
    try:
        # fetch with a bounded timeout first - pd.read_excel(url) has no timeout of its own
        # and can hang the whole cell indefinitely on a stalled connection
        req = urllib.request.Request(
            'https://archive.ics.uci.edu/ml/machine-learning-databases/00350/'
            'default%20of%20credit%20card%20clients.xls',
            headers={'User-Agent': 'aibits-course-lab/1.0'},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw = resp.read()
        df = pd.read_excel(io.BytesIO(raw), header=1, index_col=0)
        print('Loaded the real UCI dataset:', df.shape)
        return df
    except Exception as e:
        print(f'Offline fallback engaged ({e}).')
        n = 4000
        limit = np.random.lognormal(9.5, 0.6, n)
        age = np.random.randint(21, 70, n)
        bill = limit * np.random.uniform(0.1, 0.9, n)
        pay = bill * np.random.uniform(0.0, 1.0, n)
        risk = 1 / (1 + np.exp(-(-2 + 0.00002 * (bill - pay) - 0.00001 * limit)))
        default = (np.random.rand(n) < risk).astype(int)
        return pd.DataFrame({'LIMIT_BAL': limit, 'AGE': age, 'BILL_AMT1': bill,
                              'PAY_AMT1': pay, 'default payment next month': default})

df = load_credit_data()
feature_cols = [c for c in ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1'] if c in df.columns]
target_col = 'default payment next month'
X = df[feature_cols].to_numpy(dtype=np.float64)
y = df[target_col].to_numpy(dtype=np.float64).reshape(-1, 1)
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

n = len(X)
idx = np.random.permutation(n)
n_train = int(n * 0.7)
train_idx, val_idx = idx[:n_train], idx[n_train:]
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
age_val = df[['AGE']].to_numpy()[val_idx] if 'AGE' in df.columns else np.full((len(val_idx), 1), 35)
print('train:', X_train.shape, 'val:', X_val.shape)

## 2. Layers with backward (fill in the TODOs)
Each `backward(grad_out)` receives dL/d(this layer's output) and must return dL/d(this
layer's input). `Linear.backward` should also store `self.dW` and `self.db`.

In [ ]:
class Linear:
    def __init__(self, n_in, n_out):
        self.W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)  # He-ish init
        self.b = np.zeros(n_out)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, grad_out):
        # TODO: self.dW = x.T @ grad_out ; self.db = grad_out.sum(axis=0)
        # TODO: return grad_out @ W.T
        raise NotImplementedError


class ReLU:
    def forward(self, x):
        self.mask = (x > 0)
        return x * self.mask

    def backward(self, grad_out):
        # TODO: grad_out * self.mask
        raise NotImplementedError


class Sigmoid:
    def forward(self, x):
        self.out = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        return self.out

    def backward(self, grad_out):
        # TODO: grad_out * self.out * (1 - self.out)
        raise NotImplementedError


class BCELoss:
    def forward(self, y_hat, y_true, eps=1e-8):
        self.y_hat = np.clip(y_hat, eps, 1 - eps)
        self.y_true = y_true
        self.n = y_true.shape[0]
        return float(np.mean(-(y_true * np.log(self.y_hat) + (1 - y_true) * np.log(1 - self.y_hat))))

    def backward(self):
        # TODO: d(mean BCE)/d(y_hat) = (y_hat - y_true) / (y_hat * (1 - y_hat) * n)
        raise NotImplementedError

## 3. Assemble the MLP with a train step

In [ ]:
class MLP:
    def __init__(self, sizes):
        self.linears = []
        self.layers = []
        for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            lin = Linear(n_in, n_out)
            self.linears.append(lin)
            self.layers.append(lin)
            self.layers.append(Sigmoid() if i == len(sizes) - 2 else ReLU())

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def sgd_step(self, lr):
        for lin in self.linears:
            lin.W -= lr * lin.dW
            lin.b -= lr * lin.db


model = MLP([X_train.shape[1], 16, 16, 1])
loss_fn = BCELoss()

## 4. Gradient check — run this before you train anything
Compares your analytic `dW` against a finite-difference estimate on a couple of parameters.
If this fails, fix your `backward()` methods before continuing — training on broken
gradients wastes hours and is very hard to debug after the fact.

In [ ]:
def gradient_check(model, loss_fn, x, y, eps=1e-5, n_checks=5, tol=1e-4):
    y_hat = model.forward(x)
    loss_fn.forward(y_hat, y)
    model.backward(loss_fn.backward())

    lin = model.linears[0]
    W_shape = lin.W.shape
    max_err = 0.0
    for _ in range(n_checks):
        i, j = np.random.randint(W_shape[0]), np.random.randint(W_shape[1])
        orig = lin.W[i, j]

        lin.W[i, j] = orig + eps
        loss_plus = loss_fn.forward(model.forward(x), y)
        lin.W[i, j] = orig - eps
        loss_minus = loss_fn.forward(model.forward(x), y)
        lin.W[i, j] = orig

        numeric_grad = (loss_plus - loss_minus) / (2 * eps)
        y_hat = model.forward(x)
        loss_fn.forward(y_hat, y)
        model.backward(loss_fn.backward())
        analytic_grad = lin.dW[i, j]

        err = abs(numeric_grad - analytic_grad)
        max_err = max(max_err, err)
        print(f'W[{i},{j}]: analytic={analytic_grad:.6f} numeric={numeric_grad:.6f} err={err:.2e}')

    assert max_err < tol, f'Gradient check FAILED (max error {max_err:.2e} >= {tol})'
    print(f'\nGradient check PASSED (max error {max_err:.2e} < {tol})')

gradient_check(model, loss_fn, X_train[:32], y_train[:32])

## 5. Train with hand-written SGD

In [ ]:
def batches(X, y, batch_size, rng):
    idx = rng.permutation(len(X))
    for start in range(0, len(X), batch_size):
        b = idx[start:start + batch_size]
        yield X[b], y[b]

model = MLP([X_train.shape[1], 16, 16, 1])
loss_fn = BCELoss()
rng = np.random.default_rng(0)

lr, batch_size, n_epochs = 0.05, 64, 60
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    epoch_loss = 0.0
    n_batches = 0
    for xb, yb in batches(X_train, y_train, batch_size, rng):
        y_hat = model.forward(xb)
        loss = loss_fn.forward(y_hat, yb)
        model.backward(loss_fn.backward())
        model.sgd_step(lr)
        epoch_loss += loss
        n_batches += 1
    train_losses.append(epoch_loss / n_batches)
    val_losses.append(loss_fn.forward(model.forward(X_val), y_val))
    if epoch % 10 == 0 or epoch == n_epochs - 1:
        print(f'epoch {epoch:3d}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}')

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('epoch'); plt.ylabel('BCE loss'); plt.legend(); plt.title('Meridian credit-risk MLP — training curve')
plt.show()

## 6. Scorecard

In [ ]:
from sklearn.metrics import roc_auc_score

val_pred = model.forward(X_val)
auc = roc_auc_score(y_val, val_pred)
print(f'Validation AUC: {auc:.4f}  (target: beat the bank baseline of 0.77)')

young_mask = (age_val < 25).ravel()
if young_mask.sum() > 10:
    auc_young = roc_auc_score(y_val[young_mask], val_pred[young_mask])
    print(f'AUC, under-25 slice: {auc_young:.4f}  (n={young_mask.sum()}) — compare to overall AUC above')
else:
    print('Too few under-25 applicants in this split for a reliable slice metric — note this in your write-up.')

## 7. Decision-dossier-lite (fill in, ~half a page)
Does the neural network beat Meridian's gradient-boosted model by enough to justify the
added complexity, training cost, and reduced interpretability? What does the slice metric
tell you about fairness across age groups? "Not yet, and here's what would change my mind"
is a legitimate verdict.

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 2: Backprop & a tiny autograd engine*